## Device capabilities API
The CUDA API's provide functions to query values about the GPU's limits and constraints, these functions come under the device management group. For example the cudaGetDeviceProperties function will give the capabilities and properties of the GPU, it gives the essential details like the GPU name, total global memory, max threads per block amoung other capabilities. All the API functions return an error status of type cudaError_t, which can be captured for subsequent checking.
```

#include "cuda_runtime.h"

#include <stdio.h>

int main()
{
    //Get the number of GPU's on the system
    int num_of_devices;
    cudaGetDeviceCount(&num_of_devices);

    //Iterate and print each device properties
    for (int i = 0; i < num_of_devices; i++)
    {
        //Get the device i properties
        cudaDeviceProp prop;
        cudaGetDeviceProperties(&prop, i);

        printf("Device Number : %d\n", i);
        printf("Device Name : %s\n", prop.name);
        printf("Memory Clock Rate (KHz): %d\n", prop.memoryClockRate);
        printf("Memory Bus Width (bits): %d\n", prop.memoryBusWidth);
        printf("Peak Memeory Bandwidth (GB/s): %f\n\n", 2.0 * prop.memoryClockRate * (prop.memoryBusWidth / 8) / 1.0e6);

        printf("Total Global Memeory : %lu\n", prop.totalGlobalMem);
        printf("Compute Capability : %d.%d\n", prop.major, prop.minor);
        printf("Number of SMs : %d\n", prop.multiProcessorCount);
        printf("Max threads per block : %d\n", prop.maxThreadsPerBlock);
        //Max number of threads per block in each dimension(x, y, z)
        printf("Max threads dimensions : x = %d, y = %d, z = %d\n", 
                prop.maxThreadsDim[0], prop.maxThreadsDim[1], prop.maxThreadsDim[2]);
        //Max number of blocks per grid in each dimension(x, y, z)
        printf("Max grid dimensions : x = %d, y = %d, z = %d\n", 
                prop.maxGridSize[0], prop.maxGridSize[1], prop.maxGridSize[2]);
    }

    return 0;
}
```

## NVIDIA System Management Interface(nvidia-smi)
It is a command line utility provided by Nvidia, it offers monitoring and management of the GPU's in real time and gives the necessary data to do performance tuning. It can be used for 3 main features.
* Performance monitoring : Utilization, memory usage, temperature and power consumption.
* Settings management :  Controlling the clock speed and power limits.
* Device information querying : GPU name, driver version, CUDA version and current clock speeds.  

Commands
* nvidia-smi : Gives the performance metrics as 2 tables. Table 1 has the driver version, CUDA version, GPU name and performance metrics like GPU utilization, memeory usage, power usage etc. Table 2 shows the processes that are currently using the GPU.
* nvidia-smi -l 5 : Motinors the GPU continuously, refreshes the values every 5 seconds.
* nvidia-smi --query-gpu=gpu_name,driver_version,temperature.gpu --format=csv : Request only specific information, name, driver version and temperature in this case. And we are asking it to display in csv format.
* nvidia-smi pm -0 : Disable the permission mode, -1 to enable. For settings change you might need this mode to be enabled.
* nvidia-smi -i 0 -pl 150 : Set the maximum power limit to 150 watts. Only some gpu's allow changing of maximum power limit.
* nvidia-smi -q -d CLOCK : Get the current clock speeds of the different GPU core's and memory. -q stands for query. It also gives the maximum clock speeds possible for each of them. GPU atomatically lowers the clock speed if nothing is running on the GPU. We can also set the clocks to run at a contant speed using the next commands.
* nvidia-smi -q -d SUPPORTED_CLOCKS : Get all the supported clock speeds for memory and core(SM).
* nvidia-smi -application-clocks=1216,765 : Set the memory clock speed to 1216 and SM clock speed to 765, these must be in the supported clock speeds.
* nvidia-smi --reset-gpu-clocks : Reset the clock speeds to default.
* nvidia-smi -h : Get all the commands supported by nvidia-smi.

## Occupancy
Occupancy is a measure of the utilization of the resources in a GPU. It is 2 types
* Theoretical occupancy : The ideal case. It is calculated by - warp's used in a kernel/max warps per SM. We can get the maximum number of threads per SM from the GPU's white paper or from device capabilities API, dividing this by 32 will give the maximum warps per SM, say this value is 48. Now consider we are running a kernel that utilizes 48 warps/SM then occupancy is 100%, if the kernel utilizes 12 warps/SM then occupancy is 25%. To get the utilized warps/SM for a running kernel we can use Nsight compute profiler by running the command ncu {executable}. The profiler output will have a occupancy section which will have both theoretical active warps per SM and theoretical occupancy values. This refers to the ideal circumstances value, where there are enough independent tasks to fully occupy GPU's resources without being bottlenecked by memory, computation, dependency between instructions and syncronization constraints.
* Achieved occupancy : The actual usage of the GPU's resources. In the actual execution warps may be waiting for a value from the memory, during this wait other warps may be scheduled for execution. However if all the warps encounter memory requests or instruction dependency, the scheduler may encounter cycles where no warp is ready for execution, this results in stall cycles. The way this works is the warp schedular of a SM partion will ask the warp if it is ready to execute the next instruction(SIMD, same instruction on the 32 threads in the warp using 32 cores), if the warp is ready the dispatcher will take the instruction and send it to 32 cores(type of cores will depend on the instructions). If schedular finds that none of the warps are ready to execute for a given cycle, that cycle will be stalled. Stalled cycles will result in achieved occupancy being lesser than theoretical occupancy.

Hiding latency is an important feature of GPU's. In the case of CPU if 2 continuous instructions are dependent(1st instruction reading a value from memory, which takes a few cycles, 2nd instruction is using this value) then it can do out of order execution(for example execute 1st and then 3rd instructions) for better performance. This out of order execution is not possible in GPU's, instead the GPU switches to the next ready warp. By doing this and excuting many threads concurrently, GPU's mask the latency caused by instruction dependency and memory access. In a GPU, memory access can take from 30 cycles to 300 cycles depending on if it is present in L1 or L2 or global memory. Load and store are seperate units in GPU's(they get the values from memory to registers), they are not executed by the normal cores which execute the instructions. 

NVBit tool can be used to get instuction and memory traces. Nvidia GPU assembly instructions are called streaming assembler(SASS) instructions.

Lecture 27 -> 44:30